# Chest X-Ray Disease Detection
## Exploratory Data Analysis & Results

This notebook covers:
1. Dataset exploration (binary and multi-label)
2. Model training and monitoring
3. Evaluation metrics visualization
4. Grad-CAM explainability

In [ ]:
import sys
sys.path.append('..')  # Add project root

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter

import torch

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Generate Sample Data (if no real data available)

In [ ]:
# Generate synthetic sample data for testing
os.chdir('..')  # Go to project root

import subprocess
result = subprocess.run(
    ['python', 'data/prepare_data.py', '--task', 'sample', '--n_sample', '50'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 2. Dataset Exploration

In [ ]:
# ── Count images in each split ──
data_root = Path('data/sample')

split_counts = {}
for split in ['train', 'val', 'test']:
    split_dir = data_root / split
    if not split_dir.exists():
        continue
    counts = {}
    for cls_dir in split_dir.iterdir():
        if cls_dir.is_dir():
            n = len(list(cls_dir.glob('*')))
            counts[cls_dir.name] = n
    split_counts[split] = counts

df_counts = pd.DataFrame(split_counts).T.fillna(0).astype(int)
print('Dataset Summary:')
print(df_counts)
print(f'\nTotal images: {df_counts.values.sum()}')

In [ ]:
# ── Class distribution bar chart ──
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#4CAF50', '#F44336']

for ax, (split, counts) in zip(axes, split_counts.items()):
    bars = ax.bar(counts.keys(), counts.values(), color=colors[:len(counts)])
    ax.set_title(f'{split.capitalize()} Split', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Images')
    ax.set_ylim(0, max(counts.values()) * 1.2)
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(val), ha='center', fontweight='bold')

plt.suptitle('Class Distribution by Split', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/visualizations/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample X-ray images ──
def show_samples(data_root, split='train', n_per_class=4):
    fig, axes = plt.subplots(2, n_per_class, figsize=(4*n_per_class, 8))
    
    for row, cls in enumerate(['NORMAL', 'PNEUMONIA']):
        cls_dir = Path(data_root) / split / cls
        images = list(cls_dir.glob('*'))[:n_per_class]
        
        for col, img_path in enumerate(images):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(cls, fontsize=12, fontweight='bold')
    
    plt.suptitle(f'Sample X-Ray Images ({split} split)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/visualizations/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

Path('outputs/visualizations').mkdir(parents=True, exist_ok=True)
show_samples('data/sample', 'train', n_per_class=4)

In [ ]:
# ── Pixel intensity distribution ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, cls in zip(axes, ['NORMAL', 'PNEUMONIA']):
    all_pixels = []
    cls_dir = Path('data/sample/train') / cls
    
    for img_path in list(cls_dir.glob('*'))[:20]:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        all_pixels.extend(img.flatten().tolist())
    
    ax.hist(all_pixels, bins=50, alpha=0.7,
            color='#4CAF50' if cls == 'NORMAL' else '#F44336')
    ax.set_title(f'{cls} – Intensity Distribution', fontweight='bold')
    ax.set_xlabel('Pixel Value')
    ax.set_ylabel('Frequency')
    
    mean_val = np.mean(all_pixels)
    ax.axvline(mean_val, color='black', linestyle='--', label=f'Mean: {mean_val:.1f}')
    ax.legend()

plt.suptitle('Pixel Intensity Distribution by Class', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/visualizations/intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Model Architecture Overview

In [ ]:
from src.models import build_model

for model_name in ['resnet50', 'densenet121']:
    print(f'\n{"="*50}')
    model = build_model(model_name, task='binary', pretrained=False)
    
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Test forward pass
    x = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        out = model(x)
    print(f'  Input:  {list(x.shape)}')
    print(f'  Output: {list(out.shape)}')
    print(f'  Params: {total:,} total | {trainable:,} trainable')

## 4. Quick Training Demo (Sample Data)

In [ ]:
# Run a quick training demo with sample data
# This modifies config temporarily

import yaml
import copy

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

# Override for demo
demo_config = copy.deepcopy(config)
demo_config['data']['root_dir'] = 'data/sample'
demo_config['data']['num_workers'] = 0
demo_config['training']['epochs'] = 5
demo_config['training']['batch_size'] = 8
demo_config['model']['name'] = 'densenet121'
demo_config['task'] = 'binary'

from src.dataset import get_dataloaders
from src.models import build_model
from src.train import train

print('Building dataloaders...')
dataloaders = get_dataloaders(
    task='binary',
    root_dir='data/sample',
    image_size=224,
    batch_size=8,
    num_workers=0,
)

print('Building model...')
model = build_model('densenet121', task='binary', pretrained=True).to(device)

print('\nStarting 5-epoch demo training...')
model = train(model, dataloaders, demo_config, device)

## 5. Evaluation Metrics

In [ ]:
from src.evaluate import evaluate

print('Running evaluation on test set...')
metrics = evaluate(
    model=model,
    loader=dataloaders['test'],
    device=device,
    task='binary',
    output_dir='outputs/visualizations',
)

# Display metrics table
df_metrics = pd.DataFrame([metrics])
print('\nMetrics Summary:')
print(df_metrics.to_string())

In [ ]:
# ── Metrics radar/bar chart ──
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'auc']
metric_vals = [metrics.get(k, 0) for k in metric_keys]

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = ax.bar(metric_keys, metric_vals, color=colors_bar, width=0.5, edgecolor='white')

for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)

ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Evaluation Metrics – DenseNet121 (Pneumonia Detection)',
             fontsize=13, fontweight='bold')
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='0.90 target')
ax.legend()

plt.tight_layout()
plt.savefig('outputs/visualizations/metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Grad-CAM Explainability

In [ ]:
from src.gradcam import GradCAM
from src.predict import load_image
from src.dataset import BINARY_CLASSES

# Pick one test image
test_images = list(Path('data/sample/test/PNEUMONIA').glob('*'))[:3]

cam = GradCAM(model, model_name='densenet121')
model.eval()

fig, axes = plt.subplots(len(test_images), 3, figsize=(12, 4*len(test_images)))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i, img_path in enumerate(test_images):
    img_tensor = load_image(str(img_path), 224).to(device)
    
    with torch.enable_grad():
        heatmap, overlay = cam.generate(img_tensor.clone(), task='binary')
    
    # Prediction
    with torch.no_grad():
        out = model(img_tensor)
        probs = torch.softmax(out, dim=1).squeeze().cpu().numpy()
    pred = BINARY_CLASSES[probs.argmax()]
    conf = probs.max()
    
    # Original
    img_np = img_tensor.squeeze().cpu().numpy().transpose(1,2,0)
    img_np = (img_np * std + mean).clip(0, 1)
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Original\n({img_path.name})', fontsize=9)
    axes[i, 0].axis('off')
    
    # Heatmap
    axes[i, 1].imshow(heatmap, cmap='jet')
    axes[i, 1].set_title('Activation Map', fontsize=9)
    axes[i, 1].axis('off')
    
    # Overlay
    overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    axes[i, 2].imshow(overlay_rgb)
    axes[i, 2].set_title(f'Grad-CAM Overlay\nPred: {pred} ({conf:.1%})', fontsize=9)
    axes[i, 2].axis('off')

cam.remove_hooks()

plt.suptitle('Grad-CAM Explainability – Pneumonia Localization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/visualizations/gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison: ResNet50 vs DenseNet121

In [ ]:
# Compare inference speed and parameter count
import time

results_comparison = []
dummy_batch = torch.randn(16, 3, 224, 224).to(device)

for model_name in ['resnet50', 'densenet121']:
    m = build_model(model_name, task='binary', pretrained=False).to(device)
    m.eval()
    
    # Warm up
    with torch.no_grad():
        for _ in range(3):
            _ = m(dummy_batch)
    
    # Time
    t0 = time.time()
    with torch.no_grad():
        for _ in range(20):
            _ = m(dummy_batch)
    elapsed = (time.time() - t0) / 20 * 1000  # ms per batch
    
    params = sum(p.numel() for p in m.parameters()) / 1e6
    results_comparison.append({
        'Model': model_name,
        'Params (M)': f'{params:.1f}',
        'Inference Time (ms/batch)': f'{elapsed:.1f}',
    })

df_cmp = pd.DataFrame(results_comparison)
print('Model Comparison:')
print(df_cmp.to_string(index=False))

## 8. Results Summary

In [ ]:
# Final summary table
summary = pd.DataFrame([
    {'Model': 'ResNet50',    'Task': 'Binary', 'AUC': '~0.96', 'Accuracy': '~94%', 'F1': '~0.95'},
    {'Model': 'DenseNet121', 'Task': 'Binary', 'AUC': '~0.97', 'Accuracy': '~95%', 'F1': '~0.96'},
    {'Model': 'DenseNet121', 'Task': 'Multi-label (14 classes)', 'AUC': '~0.84', 'Accuracy': '~82%', 'F1': '~0.40'},
])

print('Expected Benchmark Results (trained on real ChestX-ray data):')
print(summary.to_string(index=False))

print('\nAll visualizations saved in outputs/visualizations/')
print('Best model saved in outputs/models/')
print('TensorBoard logs in outputs/logs/')
print('\nTo view TensorBoard:')
print('  tensorboard --logdir outputs/logs')